<a href="https://colab.research.google.com/github/dmitrykosintsev/lm-zoomcamp/blob/main/Homework%20files/Homework%208.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [40]:
import os
import torch
import torch.optim as optim
import torch.nn as nn
import torchvision.models as models
import numpy as np

from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torchvision import transforms
from PIL import Image

In [41]:
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

### Getting and preparing data

In [42]:
class HairClassifierCNN(nn.Module):
    def __init__(self):
        super(HairClassifierCNN, self).__init__()

        # Convolutional layer: 3 input channels -> 32 output channels
        # kernel_size=3, padding=0, stride=1
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32,
                               kernel_size=3, padding=0, stride=1)
        self.relu1 = nn.ReLU()

        # Max pooling layer with 2x2 kernel
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Calculate the flattened size after conv and pooling
        # Input: (3, 200, 200)
        # After conv (padding=0, kernel=3): (32, 198, 198)
        # After pooling (kernel=2): (32, 99, 99)
        # Flattened size: 32 * 99 * 99 = 313632

        # Fully connected layers
        self.fc1 = nn.Linear(32 * 99 * 99, 64)
        self.relu2 = nn.ReLU()

        # Output layer (no activation - will use BCEWithLogitsLoss)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        # Input shape: (batch_size, 3, 200, 200)
        x = self.conv1(x)  # -> (batch_size, 32, 198, 198)
        x = self.relu1(x)
        x = self.pool(x)    # -> (batch_size, 32, 99, 99)

        # Flatten the tensor
        x = torch.flatten(x, 1)  # -> (batch_size, 313632)

        # Fully connected layers
        x = self.fc1(x)     # -> (batch_size, 64)
        x = self.relu2(x)
        x = self.fc2(x)     # -> (batch_size, 1)

        return x


def make_model(device='cpu'):
    """
    Creates and returns the model, optimizer, and loss function.

    Args:
        device: Device to move the model to ('cpu' or 'cuda')

    Returns:
        model: The CNN model
        optimizer: SGD optimizer with lr=0.002, momentum=0.8
        criterion: BCEWithLogitsLoss for binary classification
    """
    model = HairClassifierCNN()
    model.to(device)

    optimizer = optim.SGD(model.parameters(), lr=0.002, momentum=0.8)
    criterion = nn.BCEWithLogitsLoss()

    return model, optimizer, criterion

Answer 1: nn.BCEWithLogitsLoss()

In [59]:
class HairDataset(Dataset):
    def __init__(self, data_dir, transform=None):
        self.data_dir = data_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []
        self.classes = sorted(os.listdir(data_dir))

        # Ensure binary classification (exactly 2 classes)
        assert len(self.classes) == 2, f"Expected 2 classes for binary classification, got {len(self.classes)}"

        self.class_to_idx = {cls: i for i, cls in enumerate(self.classes)}

        for label_name in self.classes:
            label_dir = os.path.join(data_dir, label_name)
            for img_name in os.listdir(label_dir):
                self.image_paths.append(os.path.join(label_dir, img_name))
                self.labels.append(self.class_to_idx[label_name])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        # Return scalar float (the unsqueeze happens in training loop)
        return image, float(label)

In [60]:
# Preparing data
# Specify our transformations
input_size = 200

# ImageNet normalization values
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

# Simple transforms - just resize and normalize
train_transforms = transforms.Compose([
    transforms.Resize((200, 200)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ) # ImageNet normalization
])

test_transforms = transforms.Compose([
    transforms.Resize((200, 200)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ) # ImageNet normalization
])

In [61]:
# Loading data
# Create dataloaders
train_dataset = HairDataset(
    data_dir='./data/train',
    transform=train_transforms
)

test_dataset = HairDataset(
    data_dir='./data/test',
    transform=test_transforms
)

# Use batch_size=20
# Use shuffle=True for both training, but False for test.
train_loader = DataLoader(train_dataset, batch_size=20, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=20, shuffle=False)

In [62]:
model, optimizer, criterion = make_model(device='cuda')

In [63]:
# Fit the model
num_epochs = 10
device='cuda'
history = {'acc': [], 'loss': [], 'val_acc': [], 'val_loss': []}

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        labels = labels.float().unsqueeze(1)  # Shape: [20] -> [20, 1]

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        predicted = (torch.sigmoid(outputs) > 0.5).float()
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_dataset)
    epoch_acc = correct_train / total_train
    history['loss'].append(epoch_loss)
    history['acc'].append(epoch_acc)

    model.eval()
    val_running_loss = 0.0
    correct_val = 0
    total_val = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            labels = labels.float().unsqueeze(1)  # Shape: [20] -> [20, 1]

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_running_loss += loss.item() * images.size(0)
            predicted = (torch.sigmoid(outputs) > 0.5).float()
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()

    val_epoch_loss = val_running_loss / len(test_dataset)
    val_epoch_acc = correct_val / total_val
    history['val_loss'].append(val_epoch_loss)
    history['val_acc'].append(val_epoch_acc)

    print(f"Epoch {epoch+1}/{num_epochs}, "
          f"Loss: {epoch_loss:.4f}, Acc: {epoch_acc:.4f}, "
          f"Val Loss: {val_epoch_loss:.4f}, Val Acc: {val_epoch_acc:.4f}")

Epoch 1/10, Loss: 0.6556, Acc: 0.5980, Val Loss: 0.6128, Val Acc: 0.6269
Epoch 2/10, Loss: 0.5703, Acc: 0.6954, Val Loss: 0.7409, Val Acc: 0.6169
Epoch 3/10, Loss: 0.6001, Acc: 0.6667, Val Loss: 0.5957, Val Acc: 0.6468
Epoch 4/10, Loss: 0.5304, Acc: 0.7079, Val Loss: 0.7057, Val Acc: 0.5771
Epoch 5/10, Loss: 0.5047, Acc: 0.7378, Val Loss: 0.5823, Val Acc: 0.6667
Epoch 6/10, Loss: 0.4622, Acc: 0.7815, Val Loss: 0.6061, Val Acc: 0.6517
Epoch 7/10, Loss: 0.3997, Acc: 0.8052, Val Loss: 0.6316, Val Acc: 0.6617
Epoch 8/10, Loss: 0.3653, Acc: 0.8464, Val Loss: 0.7680, Val Acc: 0.6119
Epoch 9/10, Loss: 0.3570, Acc: 0.8402, Val Loss: 0.6289, Val Acc: 0.7214
Epoch 10/10, Loss: 0.2540, Acc: 0.8851, Val Loss: 0.7446, Val Acc: 0.6915


In [64]:
from torchsummary import summary
summary(model, input_size=(3, 200, 200))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 32, 198, 198]             896
              ReLU-2         [-1, 32, 198, 198]               0
         MaxPool2d-3           [-1, 32, 99, 99]               0
            Linear-4                   [-1, 64]      20,072,512
              ReLU-5                   [-1, 64]               0
            Linear-6                    [-1, 1]              65
Total params: 20,073,473
Trainable params: 20,073,473
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.46
Forward/backward pass size (MB): 21.54
Params size (MB): 76.57
Estimated Total Size (MB): 98.57
----------------------------------------------------------------


Answer 2: 20073473

In [66]:
# Finding median
median_train_acc = np.median(history['acc'])
print(f"Answer 3: Median training accuracy is {median_train_acc:.4f}")

Answer 3: Median training accuracy is 0.7597


In [67]:
# STD
std_train_loss = np.std(history['loss'])
print(f"Answer 4: Standard deviation of training loss is {std_train_loss:.4f}")

Answer 4: Standard deviation of training loss is 0.1190


In [68]:
# Data augmentation
train_transforms = transforms.Compose([
    transforms.Resize((200, 200)),
    transforms.RandomRotation(50),
    transforms.RandomResizedCrop(200, scale=(0.9, 1.0), ratio=(0.9, 1.1)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ) # ImageNet normalization

])

train_dataset = HairDataset(
    data_dir='./data/train',
    transform=train_transforms
)